# 03 — Fine-tune BERTSUM (Extractive) on Colab GPU

Trains a `bert-base-uncased` encoder with a per-sentence [CLS] classifier head on the ORACLE labels from notebook 02.

In [ ]:
# Setup: clone repo + mount Drive + install deps. Needs GPU.
REPO_URL = 'https://github.com/Captain-Uchiha/scientific-paper-summarizer.git'
!rm -rf /content/scientific-summarizer
!git clone -q {REPO_URL} /content/scientific-summarizer
import sys, os
sys.path.insert(0, '/content/scientific-summarizer')
os.chdir('/content/scientific-summarizer')

!pip -q install 'transformers>=4.40' datasets accelerate tqdm

!nvidia-smi

from google.colab import drivePROJECT_DIR = '/content/drive/MyDrive/scientific-summarizer-data'
drive.mount('/content/drive', force_remount=True)

In [ ]:
import json, torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer

from models import bertsum
from models.bertsum import BertSumExt

BASE = 'bert-base-uncased'
MAX_LEN = 512
tok = AutoTokenizer.from_pretrained(BASE)

class BertSumDataset(Dataset):
    def __init__(self, path, max_len=MAX_LEN):
        self.records = []
        with open(path) as f:
            for line in f:
                self.records.append(json.loads(line))
        self.max_len = max_len
        self.cls = tok.cls_token_id; self.sep = tok.sep_token_id; self.pad = tok.pad_token_id
    def __len__(self): return len(self.records)
    def __getitem__(self, i):
        r = self.records[i]
        ids, cls_pos, lbls = [], [], []
        for sent, lab in zip(r['sentences'], r['ext_labels']):
            tids = tok.encode(sent, add_special_tokens=False)
            if len(ids) + len(tids) + 2 > self.max_len: break
            cls_pos.append(len(ids))
            ids += [self.cls] + tids + [self.sep]
            lbls.append(lab)
        attn = [1] * len(ids)
        return {
            'input_ids': torch.tensor(ids),
            'attention_mask': torch.tensor(attn),
            'cls_positions': torch.tensor(cls_pos),
            'labels': torch.tensor(lbls, dtype=torch.float),
        }

def collate(batch):
    ids  = pad_sequence([b['input_ids'] for b in batch], batch_first=True, padding_value=tok.pad_token_id)
    attn = pad_sequence([b['attention_mask'] for b in batch], batch_first=True, padding_value=0)
    max_s = max(len(b['cls_positions']) for b in batch)
    cls_pos = torch.zeros(len(batch), max_s, dtype=torch.long)
    cls_mask = torch.zeros(len(batch), max_s, dtype=torch.long)
    labels = torch.zeros(len(batch), max_s)
    for i, b in enumerate(batch):
        n = len(b['cls_positions'])
        cls_pos[i, :n] = b['cls_positions']
        cls_mask[i, :n] = 1
        labels[i, :n] = b['labels']
    return {'input_ids': ids, 'attention_mask': attn, 'cls_positions': cls_pos,
            'cls_mask': cls_mask, 'labels': labels}

PROC = f'{PROJECT_DIR}/dataset/processed'
train_ds = BertSumDataset(f'{PROC}/train.oracle.jsonl')
val_ds   = BertSumDataset(f'{PROC}/val.oracle.jsonl')
print('train/val:', len(train_ds), len(val_ds))

In [ ]:
import torch.nn as nn
from torch.optim import AdamW
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = BertSumExt(BASE).to(device)
opt = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
loss_fn = nn.BCEWithLogitsLoss(reduction='none')

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, collate_fn=collate)
val_loader   = DataLoader(val_ds, batch_size=4, shuffle=False, collate_fn=collate)

GRAD_ACC = 8
EPOCHS = 3
scaler = torch.cuda.amp.GradScaler()
CKPT_DIR = f'{PROJECT_DIR}/models/bertsum'
import os; os.makedirs(CKPT_DIR, exist_ok=True)

for epoch in range(EPOCHS):
    model.train(); opt.zero_grad()
    pbar = tqdm(train_loader, desc=f'epoch {epoch+1}')
    for step, batch in enumerate(pbar):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.cuda.amp.autocast():
            logits = model(batch['input_ids'], batch['attention_mask'],
                           batch['cls_positions'], batch['cls_mask'])
            losses = loss_fn(logits, batch['labels']) * batch['cls_mask']
            loss = losses.sum() / batch['cls_mask'].sum().clamp(min=1)
        scaler.scale(loss / GRAD_ACC).backward()
        if (step + 1) % GRAD_ACC == 0:
            scaler.step(opt); scaler.update(); opt.zero_grad()
        pbar.set_postfix(loss=float(loss))
    torch.save(model.state_dict(), f'{CKPT_DIR}/bertsum_epoch{epoch+1}.pt')
    print('saved', f'{CKPT_DIR}/bertsum_epoch{epoch+1}.pt')

In [ ]:
# Quick sanity-check: generate an extractive summary for one validation paper
model.eval()
rec = json.loads(open(f'{PROC}/val.oracle.jsonl').readline())
from preprocessing.clean import split_sentences  # noqa
summary = bertsum.bertsum_summary(rec['input_text'], model, tok, num_sentences=5)
print('PRED:\n', summary)
print('\nGOLD:\n', rec['target_text'])